In [1]:
# checking the packages
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print(f"Python executable: {sys.executable}")
print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Python executable: D:\STUDY & WORK\STUDY\MASTER\Monash University\Projects\SuperAnnuation\.venv\Scripts\python.exe
Python version: 3.12.4 (tags/v3.12.4:8e8a4ba, Jun  6 2024, 19:30:16) [MSC v.1940 64 bit (AMD64)]
pandas version: 3.0.3
NumPy version: 2.5.1


In [2]:
# establish paths and checking root and data directories
current_directory = Path.cwd()

# This works whether Jupyter starts in the project root or directly inside the notebooks directory.
if current_directory.name.lower() == "notebooks":
    PROJECT_ROOT = current_directory.parent
else:
    PROJECT_ROOT = current_directory

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "apra" / "2026-03"

print(f"Current directory: {current_directory}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Raw directory exists: {RAW_DATA_DIR.exists()}")

Current directory: D:\STUDY & WORK\STUDY\MASTER\Monash University\Projects\SuperAnnuation\notebooks
Project root: D:\STUDY & WORK\STUDY\MASTER\Monash University\Projects\SuperAnnuation
Raw data directory: D:\STUDY & WORK\STUDY\MASTER\Monash University\Projects\SuperAnnuation\data\raw\apra\2026-03
Raw directory exists: True


In [3]:
# check the raw files in the folder
raw_files = sorted(
    path
    for path in RAW_DATA_DIR.iterdir()
    if path.is_file()
)

file_inventory = pd.DataFrame(
    {
        "filename": [path.name for path in raw_files],
        "extension": [path.suffix.lower() for path in raw_files],
        "size_mb": [
            round(path.stat().st_size / (1024**2), 2)
            for path in raw_files
        ],
    }
)

file_inventory

,filename,extension,size_mb
0,2025 CPPP - Choice.xlsx,.xlsx,0.74
1,2025 CPPP - MySuper.xlsx,.xlsx,0.49
2,CPPP Methodology.pdf,.pdf,7.36
3,Glossary - Quarterly Superannuation Product Pu...,.pdf,0.30
4,Historical performance (2).csv,.csv,68.81
5,Historical SAA (2).csv,.csv,143.28
6,Quarterly fund-level statistics database versi...,.zip,1.12
7,Quarterly Superannuation Product Publication -...,.xlsx,31.98
8,Quarterly Superannuation Product Statistics - ...,.xlsx,163.20
9,Quarterly Superannuation Product Statistics - ...,.zip,103.71


For initial data analysis, see the historical performance, SAA, and the current performance first

In [4]:
def find_single_file(
    directory: Path,
    required_terms: list[str],
    extension: str,
) -> Path:
    """To find a single file based on certain terms and file extension"""
    matches = []

    for path in directory.iterdir():
        name = path.name.lower()

        if (
            path.is_file()
            and path.suffix.lower() == extension.lower()
            and all(term.lower() in name for term in required_terms)
        ):
            matches.append(path)

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Expected exactly one {extension} file containing "
            f"{required_terms}, but found: {[p.name for p in matches]}"
        )

    return matches[0]

# find the historical data file
historical_performance_path = find_single_file(
    RAW_DATA_DIR,
    required_terms=["historical", "performance"],
    extension=".csv",
)

historical_saa_path = find_single_file(
    RAW_DATA_DIR,
    required_terms=["historical", "saa"],
    extension=".csv",
)

# find current performance file
performance_workbook_path = find_single_file(
    RAW_DATA_DIR,
    required_terms=["performance", "Quarterly", "Product"],
    extension=".xlsx",
)

# find current product structure file
product_structure_path = find_single_file(
    RAW_DATA_DIR,
    required_terms=["product", "structure", "Quarterly"],
    extension=".xlsx",
)


print("Historical performance:", historical_performance_path.name)
print("Historical SAA:", historical_saa_path.name)
print("Performance workbook:", performance_workbook_path.name)
print("Product structure:", product_structure_path.name)

Historical performance: Historical performance (2).csv
Historical SAA: Historical SAA (2).csv
Performance workbook: Quarterly Superannuation Product Publication - Performance_0.xlsx
Product structure: Quarterly Superannution Product Statistics - Product Structure.xlsx


## Loading data files

### Loading historical performance file

In [5]:
# load historical performance file
historical_performance = pd.read_csv(
    historical_performance_path,
    low_memory=False,
)

# check number of rows and columns
print(f"Rows: {historical_performance.shape[0]:,}")
print(f"Columns: {historical_performance.shape[1]:,}")

historical_performance.head()

Rows: 202,664
Columns: 22


,abn_product_identifier,abn_investment_menu_identifier,abn_investment_option_identifier,fees_and_costs_arrangement_identifier,period_end_date,product_name,product_type,product_phase,investment_menu_name,investment_menu_type,...,investment_option_category,life_cycle_name,fund_name,rse_abn,trustee_name,fund_public_offer_status,open_to_new_mems,return_measurement_comparison_percent,return_investment_five_year_volatility_comparison_percent,return_investment_ten_year_volatility_comparison_percent
0,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/12/2014,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.026640,0.048896,NaN
1,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/03/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.057225,0.046906,NaN
2,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,30/06/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.016261,0.044379,NaN
3,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,30/09/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.017265,0.046924,NaN
4,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/12/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.024534,0.047364,NaN


In [6]:
# check columns
historical_performance.columns.tolist()

['abn_product_identifier',
 'abn_investment_menu_identifier',
 'abn_investment_option_identifier',
 'fees_and_costs_arrangement_identifier',
 'period_end_date',
 'product_name',
 'product_type',
 'product_phase',
 'investment_menu_name',
 'investment_menu_type',
 'investment_option_name',
 'investment_option_type',
 'investment_option_category',
 'life_cycle_name',
 'fund_name',
 'rse_abn',
 'trustee_name',
 'fund_public_offer_status',
 'open_to_new_mems',
 'return_measurement_comparison_percent',
 'return_investment_five_year_volatility_comparison_percent',
 'return_investment_ten_year_volatility_comparison_percent']

In [7]:
historical_performance.info()

# check percentage of null in each column
historical_performance.isnull().mean().round(3)*100

<class 'pandas.DataFrame'>
RangeIndex: 202664 entries, 0 to 202663
Data columns (total 22 columns):
 #   Column                                                     Non-Null Count   Dtype  
---  ------                                                     --------------   -----  
 0   abn_product_identifier                                     202664 non-null  str    
 1   abn_investment_menu_identifier                             202664 non-null  str    
 2   abn_investment_option_identifier                           202664 non-null  str    
 3   fees_and_costs_arrangement_identifier                      202664 non-null  str    
 4   period_end_date                                            202664 non-null  str    
 5   product_name                                               202664 non-null  str    
 6   product_type                                               202664 non-null  str    
 7   product_phase                                              202664 non-null  str    
 8   inves

abn_product_identifier                                         0.0
abn_investment_menu_identifier                                 0.0
abn_investment_option_identifier                               0.0
fees_and_costs_arrangement_identifier                          0.0
period_end_date                                                0.0
product_name                                                   0.0
product_type                                                   0.0
product_phase                                                  0.0
investment_menu_name                                           0.0
investment_menu_type                                           0.0
investment_option_name                                         0.0
investment_option_type                                         0.0
investment_option_category                                     0.0
life_cycle_name                                              100.0
fund_name                                                     

### Loading historical SAA file

In [8]:
# load historical saa file
historical_saa = pd.read_csv(
    historical_saa_path,
    low_memory=False,
)

# check number of rows and columns
print(f"Rows: {historical_saa.shape[0]:,}")
print(f"Columns: {historical_saa.shape[1]:,}")

historical_saa.head()

Rows: 426,917
Columns: 23


,investment_option_identifier,time_key,investment_option_name,investment_option_type,investment_option_category,rse_name,rse_abn,rsl_name,Public_Offer_Status,ConsolidatedSectorType,...,InvestmentStrategicSectorType,InvestmentStrategicSectorListingType,InvestmentStrategicSectorDomicileType,InvestmentStrategicSectorInternationalEconomyType,InvestmentStrategicSubsectorType,InvestmentStrategicSubsectorListingType,InvestmentStrategicSubsectorDomicileType,InvestmentStrategicSubsectorInternationalEconomyType,InvestmentBenchmarkAllocationPercent,InvestmentCurrencyHedgingRatioPercent
0,83810127567-AZ49C1MySuper,20140630,Balanced Growth MySuper,Multi Manager,Multi Sector,ANZ Australian Staff Superannuation Scheme,83810127567,ANZ Staff Superannuation (Australia) Pty. Limited,Non-public offer,Fixed Income,...,Fixed Income,Not Applicable,Australian Domicile,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,0.09,0.0
1,83810127567-AZ49C1MySuper,20140630,Balanced Growth MySuper,Multi Manager,Multi Sector,ANZ Australian Staff Superannuation Scheme,83810127567,ANZ Staff Superannuation (Australia) Pty. Limited,Non-public offer,Credit,...,Credit,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,0.03,1.0
2,83810127567-AZ49C1MySuper,20140630,Balanced Growth MySuper,Multi Manager,Multi Sector,ANZ Australian Staff Superannuation Scheme,83810127567,ANZ Staff Superannuation (Australia) Pty. Limited,Non-public offer,Infrastructure,...,Infrastructure,Unlisted,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,0.06,0.0
3,83810127567-AZ49C1MySuper,20140630,Balanced Growth MySuper,Multi Manager,Multi Sector,ANZ Australian Staff Superannuation Scheme,83810127567,ANZ Staff Superannuation (Australia) Pty. Limited,Non-public offer,Alternatives,...,Alternatives,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,0.08,1.0
4,83810127567-AZ49C1MySuper,20140630,Balanced Growth MySuper,Multi Manager,Multi Sector,ANZ Australian Staff Superannuation Scheme,83810127567,ANZ Staff Superannuation (Australia) Pty. Limited,Non-public offer,Cash,...,Cash,Not Applicable,Australian Domicile,Not Applicable,Not Applicable,Not Applicable,Not Applicable,Not Applicable,0.03,0.0


In [9]:
historical_saa.info()

# check percentage of null in each column
historical_saa.isnull().mean().round(3)*100

<class 'pandas.DataFrame'>
RangeIndex: 426917 entries, 0 to 426916
Data columns (total 23 columns):
 #   Column                                                Non-Null Count   Dtype  
---  ------                                                --------------   -----  
 0   investment_option_identifier                          426917 non-null  str    
 1   time_key                                              426917 non-null  int64  
 2   investment_option_name                                426917 non-null  str    
 3   investment_option_type                                426917 non-null  str    
 4   investment_option_category                            426917 non-null  str    
 5   rse_name                                              426917 non-null  str    
 6   rse_abn                                               426917 non-null  int64  
 7   rsl_name                                              426917 non-null  str    
 8   Public_Offer_Status                                   4

investment_option_identifier                            0.0
time_key                                                0.0
investment_option_name                                  0.0
investment_option_type                                  0.0
investment_option_category                              0.0
rse_name                                                0.0
rse_abn                                                 0.0
rsl_name                                                0.0
Public_Offer_Status                                     0.0
ConsolidatedSectorType                                  0.0
ConsolidatedListingType                                 0.0
ConsolidatedDomicileType                                0.0
ConsolidatedInternationalEconomyType                    0.0
InvestmentStrategicSectorType                           0.0
InvestmentStrategicSectorListingType                    0.0
InvestmentStrategicSectorDomicileType                   0.0
InvestmentStrategicSectorInternationalEc

### Loading current performance file

In [10]:
performance_excel = pd.ExcelFile(performance_workbook_path)
performance_excel.sheet_names

['Cover',
 'Notes',
 'Important notice',
 'Contents',
 'Table 2',
 'Table 3',
 'Table 4a',
 'Table 4b',
 'Table 4c',
 'Table 5a',
 'Table 5b',
 'Table 5c',
 'Table 6a',
 'Table 6b',
 'Table 6c',
 'Table 7a',
 'Table 7b',
 'Table 7c',
 'Table 7d',
 'Table 8a',
 'Table 8b',
 'Table 8c',
 'Table 8d',
 'Table 9',
 'Table 9a',
 'Table 10',
 'Table 11',
 'Explanatory notes',
 'Metrics']

In [11]:
# preview table 4a
preview_4a = pd.read_excel(
    performance_workbook_path,
    sheet_name="Table 4a",
    header=None,
    nrows=20,
)

preview_4a

,0,1,2,3,4,5,6,7,8,9,...,78,79,80,81,82,83,84,85,86,87
0,MySuper Net Returns (for representative member...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,"Total Investment, Transaction, Administration ...",NaN,Net return (Other periods),NaN,NaN,NaN,NaN,Volatility
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Pathway Identifier,Product Identifier,Investment Menu Identifier,Investment Option Identifier,Period,Superannuation Product Name,Superannuation Product Type,Investment Menu Name,Investment Menu Type,Investment Option / Lifecycle Stage Name,...,NaN,Advice-related tax expense / benefit\n (rep me...,Total Fees and Costs\n (rep member),"Total Fees, Costs and Taxes\n (rep member)",Net return \n (rep member) - Quarterly,One-year net return \n (rep member) - Annualised,Three-year net return \n (rep member) - Annual...,Five-year net return \n (rep member) - Annualised,Ten-year net return \n (rep member) - Annualised,Volatility (10 year)
5,NaN,NaN,NaN,NaN,** Refer to Metrics for further notes on calcu...,NaN,NaN,NaN,NaN,NaN,...,FC0002,FC0003,NaN,NaN,IP0011,IP0013,IP0014,IP0015,IP0016,NaN
6,NaN,NaN,NaN,NaN,NaN,SRF 605.0 Table 1 Column 3,SRF 605.0 Table 1 Column 5,SRF 605.0 Table 2 Column 3,SRF 605.0 Table 2 Column 5,SRF 605.0 Table 3 Column 3,...,SRF 705.0 Table 1 Column 15,NaN,AG+AT+BL+BY,AG+AJ+AT+AW+BL+BO+BY+CB,AX-BL-BO-BY-CB,NaN,NaN,NaN,NaN,SRF 705.1 Table 2 column 12
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,(%),(%),(%),(%),(%),(%),(%),(%),(%),(%)
8,85502108833-SPITS1600-IMITS1100-IOITS1000,85502108833-SPITS1600,85502108833-IMITS1100,85502108833-IOITS1000,2025-03-31,TelstraSuper MySuper,MySuper Product,MySuper,Lifecycle Option,MySuper Growth,...,0,0,0.002629,0.000422,-0.010805,0.0514,0.0634,0.1036,0.0696,0.077021
9,60905115063-60905115063329-60905115063329-X0,60905115063-60905115063329,60905115063-60905115063329,60905115063-X0,2025-03-31,QSuper Lifetime,MySuper Product,QSuper Lifetime,Lifecycle Option,Sustain 3 Group,...,0,0,0.001344,0.001021,-0.000282,0.051,0.037,0.0357,0.0317,NaN


In [12]:
from openpyxl.utils import get_column_letter
import re

header_block_4a = pd.read_excel(
    performance_workbook_path,
    sheet_name="Table 4a",
    header=None,
    skiprows=4,
    nrows=4,
)

performance_4a = pd.read_excel(
    performance_workbook_path,
    sheet_name="Table 4a",
    header=None,
    skiprows=8,
)


header_block_4a = header_block_4a.iloc[:, :performance_4a.shape[1]]

In [13]:
def clean_header_text(value: object) -> str:
    """Convert an Excel header value to clean text."""
    if pd.isna(value):
        return ""

    return re.sub(r"\s+", " ", str(value)).strip()

metric_names = (
    header_block_4a
    .iloc[0]
    .ffill()
    .map(clean_header_text)
)

metric_codes = (
    header_block_4a
    .iloc[1]
    .map(clean_header_text)
)

source_references = (
    header_block_4a
    .iloc[2]
    .map(clean_header_text)
)

units = (
    header_block_4a
    .iloc[3]
    .map(clean_header_text)
)

In [14]:
def normalise_component(value: object) -> str:
    """Convert a header component into a clean snake_case string."""
    text = clean_header_text(value).lower()

    unit_mapping = {
        "($)": "aud",
        "$": "aud",
        "(%)": "percent",
        "%": "percent",
    }

    if text in unit_mapping:
        return unit_mapping[text]

    text = re.sub(r"[^a-z0-9]+", "_", text)

    return text.strip("_")

base_column_names = []

for metric_name, metric_code, unit in zip(
    metric_names,
    metric_codes,
    units,
):
    parts = [
        normalise_component(metric_name),
        normalise_component(metric_code),
        normalise_component(unit),
    ]

    parts = [part for part in parts if part]

    base_column_names.append("__".join(parts))

In [15]:
def make_unique_column_names(
    metric_names: pd.Series,
    metric_codes: pd.Series,
    source_references: pd.Series,
    units: pd.Series,
) -> list[str]:
    """Generate stable and unique names from APRA header metadata."""

    names = []
    name_counts = {}

    for position, (
        metric_name,
        metric_code,
        source_reference,
        unit,
    ) in enumerate(
        zip(
            metric_names,
            metric_codes,
            source_references,
            units,
        )
    ):
        components = [
            normalise_component(metric_name),
            normalise_component(metric_code),
            normalise_component(unit),
        ]

        components = [
            component
            for component in components
            if component
        ]

        base_name = "__".join(components)

        if not base_name:
            base_name = f"unnamed_column_{position + 1}"

        if base_name not in name_counts:
            name_counts[base_name] = 1
            final_name = base_name
        else:
            name_counts[base_name] += 1

            source_component = normalise_component(source_reference)
            excel_column = get_column_letter(position + 1).lower()

            if source_component:
                candidate = (
                    f"{base_name}__{source_component}"
                )
            else:
                candidate = (
                    f"{base_name}__excel_{excel_column}"
                )

            # Final safeguard in case source references also repeat.
            if candidate in names:
                candidate = (
                    f"{candidate}__{name_counts[base_name]}"
                )

            final_name = candidate

        names.append(final_name)

    return names


flat_column_names = make_unique_column_names(
    metric_names=metric_names,
    metric_codes=metric_codes,
    source_references=source_references,
    units=units,
)

performance_4a.columns = flat_column_names

assert performance_4a.columns.is_unique

In [16]:
performance_4a = performance_4a.rename(columns={"period__refer_to_metrics_for_further_notes_on_calculations":"period"})
performance_4a.head()

,pathway_identifier,product_identifier,investment_menu_identifier,investment_option_identifier,period,superannuation_product_name,superannuation_product_type,investment_menu_name,investment_menu_type,investment_option_lifecycle_stage_name,...,advice_related_tax_expense_benefit__fc0002__percent,advice_related_tax_expense_benefit_rep_member__fc0003__percent,total_fees_and_costs_rep_member__percent,total_fees_costs_and_taxes_rep_member__percent,net_return_rep_member_quarterly__ip0011__percent,one_year_net_return_rep_member_annualised__ip0013__percent,three_year_net_return_rep_member_annualised__ip0014__percent,five_year_net_return_rep_member_annualised__ip0015__percent,ten_year_net_return_rep_member_annualised__ip0016__percent,volatility_10_year__percent
0,85502108833-SPITS1600-IMITS1100-IOITS1000,85502108833-SPITS1600,85502108833-IMITS1100,85502108833-IOITS1000,2025-03-31,TelstraSuper MySuper,MySuper Product,MySuper,Lifecycle Option,MySuper Growth,...,0,0,0.002629,0.000422,-0.010805,0.0514,0.0634,0.1036,0.0696,0.077021
1,60905115063-60905115063329-60905115063329-X0,60905115063-60905115063329,60905115063-60905115063329,60905115063-X0,2025-03-31,QSuper Lifetime,MySuper Product,QSuper Lifetime,Lifecycle Option,Sustain 3 Group,...,0,0,0.001344,0.001021,-0.000282,0.0510,0.0370,0.0357,0.0317,NaN
2,60905115063-SSLIS-SUNLIS-LIS1,60905115063-SSLIS,60905115063-SUNLIS,60905115063-LIS1,2025-03-31,Lifecycle Investment Strategy,MySuper Product,Super Savings MySuper,Lifecycle Option,Less than 50,...,0,0,0.002072,-0.000301,-0.008988,0.0598,0.0663,0.0996,0.0738,0.059678
3,60905115063-SSLIS-SUNLIS-LIS16,60905115063-SSLIS,60905115063-SUNLIS,60905115063-LIS16,2025-03-31,Lifecycle Investment Strategy,MySuper Product,Super Savings MySuper,Lifecycle Option,Age 53 to 54,...,0,0,0.002017,-0.000024,-0.007230,0.0588,0.0660,0.0994,0.0737,0.059441
4,60905115063-60905115063329-60905115063329-X9,60905115063-60905115063329,60905115063-60905115063329,60905115063-X9,2025-03-31,QSuper Lifetime,MySuper Product,QSuper Lifetime,Lifecycle Option,Outlook Group,...,0,0,0.001942,-0.002828,-0.007073,0.0619,0.0448,0.0708,0.0631,0.054127


In [17]:
column_dictionary_performance_4a = pd.DataFrame(
    {
        "column_position": range(performance_4a.shape[1]),
        "excel_column": [
            get_column_letter(index + 1)
            for index in range(performance_4a.shape[1])
        ],
        "metric_name": metric_names.to_list(),
        "metric_code": metric_codes.to_list(),
        "source_reference": source_references.to_list(),
        "unit": units.to_list(),
        "clean_column_name": flat_column_names,
    }
)

display(column_dictionary_performance_4a)

column_dictionary_performance_4a[
    (column_dictionary_performance_4a["metric_code"] == "")
    | (column_dictionary_performance_4a["source_reference"] == "")
]

,column_position,excel_column,metric_name,metric_code,source_reference,unit,clean_column_name
0,0,A,Pathway Identifier,,,,pathway_identifier
1,1,B,Product Identifier,,,,product_identifier
2,2,C,Investment Menu Identifier,,,,investment_menu_identifier
3,3,D,Investment Option Identifier,,,,investment_option_identifier
4,4,E,Period,** Refer to Metrics for further notes on calcu...,,,period__refer_to_metrics_for_further_notes_on_...
...,...,...,...,...,...,...,...
83,83,CF,One-year net return (rep member) - Annualised,IP0013,,(%),one_year_net_return_rep_member_annualised__ip0...
84,84,CG,Three-year net return (rep member) - Annualised,IP0014,,(%),three_year_net_return_rep_member_annualised__i...
85,85,CH,Five-year net return (rep member) - Annualised,IP0015,,(%),five_year_net_return_rep_member_annualised__ip...
86,86,CI,Ten-year net return (rep member) - Annualised,IP0016,,(%),ten_year_net_return_rep_member_annualised__ip0...


,column_position,excel_column,metric_name,metric_code,source_reference,unit,clean_column_name
0,0,A,Pathway Identifier,,,,pathway_identifier
1,1,B,Product Identifier,,,,product_identifier
2,2,C,Investment Menu Identifier,,,,investment_menu_identifier
3,3,D,Investment Option Identifier,,,,investment_option_identifier
4,4,E,Period,** Refer to Metrics for further notes on calcu...,,,period__refer_to_metrics_for_further_notes_on_...
5,5,F,Superannuation Product Name,,SRF 605.0 Table 1 Column 3,,superannuation_product_name
6,6,G,Superannuation Product Type,,SRF 605.0 Table 1 Column 5,,superannuation_product_type
7,7,H,Investment Menu Name,,SRF 605.0 Table 2 Column 3,,investment_menu_name
8,8,I,Investment Menu Type,,SRF 605.0 Table 2 Column 5,,investment_menu_type
9,9,J,Investment Option / Lifecycle Stage Name,,SRF 605.0 Table 3 Column 3,,investment_option_lifecycle_stage_name


In [30]:
performance_4a.columns

Index(['pathway_identifier', 'product_identifier',
       'investment_menu_identifier', 'investment_option_identifier', 'period',
       'superannuation_product_name', 'superannuation_product_type',
       'investment_menu_name', 'investment_menu_type',
       'investment_option_lifecycle_stage_name', 'investment_option_type',
       'investment_option_category', 'lifecycle_strategy_indicator',
       'number_of_lifecycle_stages', 'rse_name', 'rse_abn', 'rse_licensee',
       'rse_public_offer_status',
       'member_accounts_mysuper_rounded_to_nearest_10',
       'proportion_of_member_accounts_in_lifecycle_stage__percent',
       'total_member_assets__000',
       'proportion_of_member_assets_in_lifecycle_stage__percent',
       'gross_investment_return_quarterly__percent',
       'investment_fees_and_costs__fc0001__aud',
       'investment_fees_and_costs__fc0002__percent',
       'investment_fees_and_costs_rep_member__fc0003__percent',
       'investment_costs_indirect_costs_icr__fc0

#### load performance table 8a, 9, 9a

In [18]:
from typing import Any

In [19]:
IGNORED_SECONDARY_LABELS = {
    "refer to metrics for further notes on calculations",
}


def clean_header_text(value: Any) -> str:
    """Convert an Excel header value to clean single-line text."""
    if pd.isna(value):
        return ""

    return re.sub(r"\s+", " ", str(value)).strip()


def should_ignore_secondary_label(value: Any) -> bool:
    """
    Identify explanatory row-6 text that should not become part
    of a generated column name.
    """
    text = clean_header_text(value)

    comparison_text = re.sub(
        r"^\*+\s*",
        "",
        text,
    ).strip().casefold()

    return any(
        comparison_text.startswith(ignored_label)
        for ignored_label in IGNORED_SECONDARY_LABELS
    )


def clean_secondary_label(value: Any) -> str:
    """
    Clean a secondary header label.

    Explanatory notes such as:
    '** Refer to Metrics for further notes on calculations'
    are converted to an empty string.
    """
    if should_ignore_secondary_label(value):
        return ""

    return clean_header_text(value)


def normalise_column_component(value: Any) -> str:
    """Convert a header component into snake_case."""
    text = clean_header_text(value)

    if not text:
        return ""

    unit_mapping = {
        "$": "aud",
        "($)": "aud",
        "aud": "aud",
        "%": "percent",
        "(%)": "percent",
        "percentage": "percent",
    }

    casefolded_text = text.casefold()

    if casefolded_text in unit_mapping:
        return unit_mapping[casefolded_text]

    normalised_text = re.sub(
        r"[^a-z0-9]+",
        "_",
        casefolded_text,
    )

    return normalised_text.strip("_")

# check metadata row
def read_excel_metadata_row(
    workbook_path: Path,
    sheet_name: str,
    excel_row: int | None,
    column_count: int,
) -> pd.Series:
    """
    Read one Excel row and align it to the number of data columns.

    When excel_row is None, return an empty metadata series.
    """
    if excel_row is None:
        return pd.Series(
            [""] * column_count,
            dtype="object",
        )

    metadata_row = pd.read_excel(
        workbook_path,
        sheet_name=sheet_name,
        header=None,
        skiprows=excel_row - 1,
        nrows=1,
    )

    if metadata_row.empty:
        raise ValueError(
            f"{sheet_name}: Excel row {excel_row} could not be read."
        )

    # Reindex ensures that metadata and data have exactly
    # the same number of columns.
    metadata_values = (
        metadata_row
        .iloc[0]
        .reindex(range(column_count))
    )

    return metadata_values

# unique column name construction
def make_unique_column_names(
    metric_names: pd.Series,
    secondary_labels: pd.Series,
    source_references: pd.Series,
    units: pd.Series,
) -> list[str]:
    """
    Generate unique column names from available APRA metadata.

    Main name components:
    - metric name;
    - secondary label, where applicable;
    - unit.

    The source reference and Excel column letter are used only
    to resolve remaining duplicates.
    """
    generated_names: list[str] = []
    base_name_counts: dict[str, int] = {}

    for position, (
        metric_name,
        secondary_label,
        source_reference,
        unit,
    ) in enumerate(
        zip(
            metric_names,
            secondary_labels,
            source_references,
            units,
        )
    ):
        components = [
            normalise_column_component(metric_name),
            normalise_column_component(secondary_label),
            normalise_column_component(unit),
        ]

        components = [
            component
            for component in components
            if component
        ]

        base_name = "__".join(components)

        if not base_name:
            base_name = f"unnamed_column_{position + 1}"

        base_name_counts[base_name] = (
            base_name_counts.get(base_name, 0) + 1
        )

        if base_name_counts[base_name] == 1:
            candidate_name = base_name

        else:
            source_component = normalise_column_component(
                source_reference
            )

            excel_column = get_column_letter(
                position + 1
            ).casefold()

            if source_component:
                candidate_name = (
                    f"{base_name}__{source_component}"
                )
            else:
                candidate_name = (
                    f"{base_name}__excel_{excel_column}"
                )

            suffix_number = 2

            while candidate_name in generated_names:
                candidate_name = (
                    f"{base_name}"
                    f"__excel_{excel_column}"
                    f"__{suffix_number}"
                )
                suffix_number += 1

        generated_names.append(candidate_name)

    return generated_names

def load_performance_sheet(
    workbook_path: Path,
    sheet_name: str,
    metric_row_excel: int,
    source_reference_row_excel: int,
    unit_row_excel: int,
    data_start_row_excel: int,
    secondary_row_excel: int | None = None,
) -> dict[str, pd.DataFrame]:
    """
    Load one APRA performance workbook sheet.

    The secondary metadata row is optional because Tables 9 and 9a
    do not contain one.

    Returns:
    - raw: imported observations with generated column names;
    - audit: raw observations plus audit flags;
    - clean: derived data excluding completely blank rows;
    - column_dictionary: metadata for every source column.
    """

    # Import data exactly from its stated first data row.
    performance_values = pd.read_excel(
        workbook_path,
        sheet_name=sheet_name,
        header=None,
        skiprows=data_start_row_excel - 1,
    )

    column_count = performance_values.shape[1]

    if column_count == 0:
        raise ValueError(
            f"{sheet_name}: no data columns were imported."
        )

    # Read each metadata row independently.
    metric_names = read_excel_metadata_row(
        workbook_path=workbook_path,
        sheet_name=sheet_name,
        excel_row=metric_row_excel,
        column_count=column_count,
    )

    secondary_labels = read_excel_metadata_row(
        workbook_path=workbook_path,
        sheet_name=sheet_name,
        excel_row=secondary_row_excel,
        column_count=column_count,
    )

    source_references = read_excel_metadata_row(
        workbook_path=workbook_path,
        sheet_name=sheet_name,
        excel_row=source_reference_row_excel,
        column_count=column_count,
    )

    units = read_excel_metadata_row(
        workbook_path=workbook_path,
        sheet_name=sheet_name,
        excel_row=unit_row_excel,
        column_count=column_count,
    )

    # Forward-fill only the metric-name row because merged Excel
    # metric headings leave subsequent cells blank.
    metric_names = (
        metric_names
        .ffill()
        .map(clean_header_text)
    )

    # Never forward-fill the optional secondary labels,
    # source references or units.
    secondary_labels = secondary_labels.map(
        clean_secondary_label
    )

    source_references = source_references.map(
        clean_header_text
    )

    units = units.map(
        clean_header_text
    )

    clean_column_names = make_unique_column_names(
        metric_names=metric_names,
        secondary_labels=secondary_labels,
        source_references=source_references,
        units=units,
    )

    if len(clean_column_names) != column_count:
        raise ValueError(
            f"{sheet_name}: expected {column_count} generated names, "
            f"but generated {len(clean_column_names)}."
        )

    if len(set(clean_column_names)) != len(clean_column_names):
        raise ValueError(
            f"{sheet_name}: generated column names are not unique."
        )

    performance_raw = performance_values.copy()
    performance_raw.columns = clean_column_names

    # Preserve the original source row for traceability.
    performance_raw.insert(
        0,
        "source_excel_row",
        range(
            data_start_row_excel,
            data_start_row_excel + len(performance_raw),
        ),
    )

    performance_column_dictionary = pd.DataFrame(
        {
            "sheet_name": sheet_name,
            "column_position": range(column_count),
            "excel_column": [
                get_column_letter(position + 1)
                for position in range(column_count)
            ],
            "metric_name": metric_names.to_list(),
            "secondary_label": secondary_labels.to_list(),
            "source_reference": source_references.to_list(),
            "unit": units.to_list(),
            "clean_column_name": clean_column_names,
        }
    )

    # Flag completely blank imported rows without changing raw data.
    source_data_columns = performance_raw.columns.drop(
        "source_excel_row"
    )

    performance_audit = performance_raw.copy()

    performance_audit["is_completely_blank"] = (
        performance_audit[source_data_columns]
        .isna()
        .all(axis=1)
    )

    # Derived working version only. Raw and audit versions remain intact.
    performance_clean = (
        performance_audit
        .loc[
            ~performance_audit["is_completely_blank"]
        ]
        .copy()
        .reset_index(drop=True)
    )

    return {
        "raw": performance_raw,
        "audit": performance_audit,
        "clean": performance_clean,
        "column_dictionary": performance_column_dictionary,
    }

In [20]:
performance_sheet_configs = {
    "8a": {
        "sheet_name": "Table 8a",
        "metric_row_excel": 5,
        "secondary_row_excel": 6,
        "source_reference_row_excel": 7,
        "unit_row_excel": 8,
        "data_start_row_excel": 9,
    },
    "9": {
        "sheet_name": "Table 9",
        "metric_row_excel": 5,
        "secondary_row_excel": None,
        "source_reference_row_excel": 6,
        "unit_row_excel": 7,
        "data_start_row_excel": 8,
    },
    "9a": {
        "sheet_name": "Table 9a",
        "metric_row_excel": 5,
        "secondary_row_excel": None,
        "source_reference_row_excel": 6,
        "unit_row_excel": 7,
        "data_start_row_excel": 8,
    },
}

# load the data in each table
performance_loaded_sheets = {}

for sheet_key, sheet_config in performance_sheet_configs.items():
    performance_loaded_sheets[sheet_key] = (
        load_performance_sheet(
            workbook_path=performance_workbook_path,
            **sheet_config,
        )
    )

performance_8a_raw = performance_loaded_sheets["8a"]["raw"]
performance_8a_audit = performance_loaded_sheets["8a"]["audit"]
performance_8a_clean = performance_loaded_sheets["8a"]["clean"]
performance_8a_column_dictionary = (
    performance_loaded_sheets["8a"]["column_dictionary"]
)

performance_9_raw = performance_loaded_sheets["9"]["raw"]
performance_9_audit = performance_loaded_sheets["9"]["audit"]
performance_9_clean = performance_loaded_sheets["9"]["clean"]
performance_9_column_dictionary = (
    performance_loaded_sheets["9"]["column_dictionary"]
)

performance_9a_raw = performance_loaded_sheets["9a"]["raw"]
performance_9a_audit = performance_loaded_sheets["9a"]["audit"]
performance_9a_clean = performance_loaded_sheets["9a"]["clean"]
performance_9a_column_dictionary = (
    performance_loaded_sheets["9a"]["column_dictionary"]
)

In [21]:
performance_column_dictionaries = {
    "8a": performance_8a_column_dictionary,
    "9": performance_9_column_dictionary,
    "9a": performance_9a_column_dictionary,
}

for sheet_key, dictionary in performance_column_dictionaries.items():
    metrics_note_rows = dictionary[
        dictionary["secondary_label"]
        .str.contains(
            "refer to metrics",
            case=False,
            na=False,
        )
    ]

    print(
        f"Table {sheet_key}: "
        f"{len(metrics_note_rows)} unfiltered Metrics notes"
    )


# check table 9 and 9a
assert (
    performance_9_column_dictionary["secondary_label"]
    .eq("")
    .all()
)

assert (
    performance_9a_column_dictionary["secondary_label"]
    .eq("")
    .all()
)

Table 8a: 0 unfiltered Metrics notes
Table 9: 0 unfiltered Metrics notes
Table 9a: 0 unfiltered Metrics notes


In [22]:
for sheet_key, dictionary in performance_column_dictionaries.items():
    invalid_names = dictionary[
        dictionary["clean_column_name"]
        .str.contains(
            "refer_to_metrics",
            case=False,
            na=False,
        )
    ]

    assert invalid_names.empty, (
        f"Table {sheet_key} contains Metrics-note text "
        "inside generated column names."
    )

performance_summary = pd.DataFrame(
    [
        {
            "sheet": "Table 8a",
            "raw_rows": len(performance_8a_raw),
            "clean_rows": len(performance_8a_clean),
            "columns": performance_8a_raw.shape[1],
            "blank_rows": int(
                performance_8a_audit[
                    "is_completely_blank"
                ].sum()
            ),
        },
        {
            "sheet": "Table 9",
            "raw_rows": len(performance_9_raw),
            "clean_rows": len(performance_9_clean),
            "columns": performance_9_raw.shape[1],
            "blank_rows": int(
                performance_9_audit[
                    "is_completely_blank"
                ].sum()
            ),
        },
        {
            "sheet": "Table 9a",
            "raw_rows": len(performance_9a_raw),
            "clean_rows": len(performance_9a_clean),
            "columns": performance_9a_raw.shape[1],
            "blank_rows": int(
                performance_9a_audit[
                    "is_completely_blank"
                ].sum()
            ),
        },
    ]
)

display(performance_summary)

,sheet,raw_rows,clean_rows,columns,blank_rows
0,Table 8a,1718,1718,130,0
1,Table 9,16822,16822,37,0
2,Table 9a,2601,2601,32,0


In [23]:
display(performance_8a_column_dictionary)
display(performance_9_column_dictionary)
display(performance_9a_column_dictionary)

,sheet_name,column_position,excel_column,metric_name,secondary_label,source_reference,unit,clean_column_name
0,Table 8a,0,A,Investment Option Identifier,,,,investment_option_identifier
1,Table 8a,1,B,Period,,,,period
2,Table 8a,2,C,MySuper Product Name,,SRF 605.0 Table 1 Column 3,,mysuper_product_name
3,Table 8a,3,D,MySuper product type,,SRF 605.0 Table 1 Column 6,,mysuper_product_type
4,Table 8a,4,E,Investment Option / Lifecycle Stage Name,,SRF 605.0 Table 3 Column 3,,investment_option_lifecycle_stage_name
...,...,...,...,...,...,...,...,...
124,Table 8a,124,DU,Defensive Alternatives - Lower end of asset al...,,SRF 550.0 Table 1 Column 11,(%),defensive_alternatives_lower_end_of_asset_allo...
125,Table 8a,125,DV,Defensive Alternatives - Upper end of asset al...,,SRF 550.0 Table 1 Column 12,(%),defensive_alternatives_upper_end_of_asset_allo...
126,Table 8a,126,DW,Currency Exposure (Option Level),,SRF 550.0 Table 1 Column 13,(%),currency_exposure_option_level__percent
127,Table 8a,127,DX,Growth asset weighting,,,(%),growth_asset_weighting__percent


,sheet_name,column_position,excel_column,metric_name,secondary_label,source_reference,unit,clean_column_name
0,Table 9,0,A,Pathway Identifier,,,,pathway_identifier
1,Table 9,1,B,Product Identifier,,,,product_identifier
2,Table 9,2,C,Investment Menu Identifier,,,,investment_menu_identifier
3,Table 9,3,D,Investment Option Identifier,,,,investment_option_identifier
4,Table 9,4,E,Period,,,,period
5,Table 9,5,F,Superannuation Product Name,,,,superannuation_product_name
6,Table 9,6,G,Superannuation Product Type,,,,superannuation_product_type
7,Table 9,7,H,Product Phase,,SRF 605.0 Table 1 Column 8,,product_phase
8,Table 9,8,I,Investment Menu Name,,SRF 605.0 Table 2 Column 3,,investment_menu_name
9,Table 9,9,J,Investment Menu Type,,,,investment_menu_type


,sheet_name,column_position,excel_column,metric_name,secondary_label,source_reference,unit,clean_column_name
0,Table 9a,0,A,Pathway Identifier,,,,pathway_identifier
1,Table 9a,1,B,Period,,,,period
2,Table 9a,2,C,Superannuation Product Name,,,,superannuation_product_name
3,Table 9a,3,D,Superannuation Product Type,,,,superannuation_product_type
4,Table 9a,4,E,Product Phase,,SRF 605.0 Table 1 Column 8,,product_phase
5,Table 9a,5,F,Investment Menu Name,,SRF 605.0 Table 2 Column 3,,investment_menu_name
6,Table 9a,6,G,Investment Menu Type,,,,investment_menu_type
7,Table 9a,7,H,Investment Option / Lifecycle Stage Name,,SRF 605.0 Table 2 Column 3,,investment_option_lifecycle_stage_name
8,Table 9a,8,I,Investment Option Type,,SRF 605.0 Table 2 Column 5,,investment_option_type
9,Table 9a,9,J,Investment Option Category,,SRF 605.0 Table 3 Column 3,,investment_option_category


### Loading product structure file

In [24]:
structure_excel = pd.ExcelFile(product_structure_path)
structure_excel.sheet_names

['Cover',
 'Notes',
 'Important notice',
 'Contents',
 'Table 1a',
 'Table 1b',
 'Explanatory notes',
 'Metrics']

In [25]:
structure_sheet_configs = {
    "1a": {
        "sheet_name": "Table 1a",
        "metric_row_excel": 4,
        "secondary_row_excel": None,
        "source_reference_row_excel": 5,
        "unit_row_excel": 6,
        "data_start_row_excel": 7,
    },
    "1b": {
        "sheet_name": "Table 1b",
        "metric_row_excel": 4,
        "secondary_row_excel": None,
        "source_reference_row_excel": 5,
        "unit_row_excel": 6,
        "data_start_row_excel": 7,
    },
}

# load the data in each table
structure_loaded_sheets = {}

for sheet_key, sheet_config in structure_sheet_configs.items():
    structure_loaded_sheets[sheet_key] = (
        load_performance_sheet(
            workbook_path=product_structure_path,
            **sheet_config,
        )
    )

structure_1a_raw = structure_loaded_sheets["1a"]["raw"]
structure_1a_audit = structure_loaded_sheets["1a"]["audit"]
structure_1a_clean = structure_loaded_sheets["1a"]["clean"]
structure_1a_column_dictionary = (
    structure_loaded_sheets["1a"]["column_dictionary"]
)

structure_1b_raw = structure_loaded_sheets["1b"]["raw"]
structure_1b_audit = structure_loaded_sheets["1b"]["audit"]
structure_1b_clean = structure_loaded_sheets["1b"]["clean"]
structure_1b_column_dictionary = (
    structure_loaded_sheets["1b"]["column_dictionary"]
)

In [26]:
structure_column_dictionaries = {
    "1a": structure_1a_column_dictionary,
    "1b": structure_1b_column_dictionary,
}

for sheet_key, dictionary in structure_column_dictionaries.items():
    metrics_note_rows = dictionary[
        dictionary["secondary_label"]
        .str.contains(
            "refer to metrics",
            case=False,
            na=False,
        )
    ]

    print(
        f"Table {sheet_key}: "
        f"{len(metrics_note_rows)} unfiltered Metrics notes"
    )


# check table 1a and 1b
assert (
    structure_1a_column_dictionary["secondary_label"]
    .eq("")
    .all()
)

assert (
    structure_1b_column_dictionary["secondary_label"]
    .eq("")
    .all()
)

Table 1a: 0 unfiltered Metrics notes
Table 1b: 0 unfiltered Metrics notes


In [27]:
structure_summary = pd.DataFrame(
    [
        {
            "sheet": "Table 1a",
            "raw_rows": len(structure_1a_raw),
            "clean_rows": len(structure_1a_clean),
            "columns": structure_1a_raw.shape[1],
            "blank_rows": int(
                structure_1a_audit[
                    "is_completely_blank"
                ].sum()
            ),
        },
        {
            "sheet": "Table 1b",
            "raw_rows": len(structure_1b_raw),
            "clean_rows": len(structure_1b_clean),
            "columns": structure_1b_raw.shape[1],
            "blank_rows": int(
                structure_1b_audit[
                    "is_completely_blank"
                ].sum()
            ),
        },
    ]
)

display(structure_summary)

,sheet,raw_rows,clean_rows,columns,blank_rows
0,Table 1a,830,830,24,0
1,Table 1b,130288,130288,41,0


## Checking data files

### Checking historical performance file

In [28]:
historical_performance.head()

,abn_product_identifier,abn_investment_menu_identifier,abn_investment_option_identifier,fees_and_costs_arrangement_identifier,period_end_date,product_name,product_type,product_phase,investment_menu_name,investment_menu_type,...,investment_option_category,life_cycle_name,fund_name,rse_abn,trustee_name,fund_public_offer_status,open_to_new_mems,return_measurement_comparison_percent,return_investment_five_year_volatility_comparison_percent,return_investment_ten_year_volatility_comparison_percent
0,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/12/2014,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.026640,0.048896,NaN
1,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/03/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.057225,0.046906,NaN
2,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,30/06/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.016261,0.044379,NaN
3,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,30/09/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.017265,0.046924,NaN
4,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,31/12/2015,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.024534,0.047364,NaN


In [29]:
historical_performance.info()

# check percentage of null in each column
historical_performance.isnull().mean().round(3)*100

<class 'pandas.DataFrame'>
RangeIndex: 202664 entries, 0 to 202663
Data columns (total 22 columns):
 #   Column                                                     Non-Null Count   Dtype  
---  ------                                                     --------------   -----  
 0   abn_product_identifier                                     202664 non-null  str    
 1   abn_investment_menu_identifier                             202664 non-null  str    
 2   abn_investment_option_identifier                           202664 non-null  str    
 3   fees_and_costs_arrangement_identifier                      202664 non-null  str    
 4   period_end_date                                            202664 non-null  str    
 5   product_name                                               202664 non-null  str    
 6   product_type                                               202664 non-null  str    
 7   product_phase                                              202664 non-null  str    
 8   inves

abn_product_identifier                                         0.0
abn_investment_menu_identifier                                 0.0
abn_investment_option_identifier                               0.0
fees_and_costs_arrangement_identifier                          0.0
period_end_date                                                0.0
product_name                                                   0.0
product_type                                                   0.0
product_phase                                                  0.0
investment_menu_name                                           0.0
investment_menu_type                                           0.0
investment_option_name                                         0.0
investment_option_type                                         0.0
investment_option_category                                     0.0
life_cycle_name                                              100.0
fund_name                                                     

Based on the column information, life_cycle_name could be ignored because most of the data is NULL or empty. Currently there's no way to impute or fill the column as there were no reference we can use.

But the current performance file has the life cycle column. Maybe we can use that to fill the column since the data should be a categorical or at least a string-type column.

return_investment_five_year_volatility_comparison_percent has 23.9% NULL values. Even though it is less than half, it is still quite large therefore we might have to find a way to handle it.
return_investment_ten_year_volatility_comparison_percent has 41.7% NULL values. It is significantly larger than five year metric, and it might have to be handled with care.
Both columns and return_measurement_comparison_person names don't exist in the current performance column. This is a problem because it is hard to integrate current performance to historical performance, and current performance file has more detailed metrics compared to historical performance. Not only that, the main difference is there is no "comparison" columns in the current performance file. 

Based on the glossary, return_investment_ten_year_volatility_comparison_percent have the same definition as volatility (10 year) in current performance, therefore it can be combined later on.

Based on the glossary, return_measurement_comparison_person might be equivalent to NIR percentage in Table 4a.

#### Convert end period into date field

In [31]:
historical_performance["period_end_date"] = pd.to_datetime(
    historical_performance["period_end_date"],
    format="%d/%m/%Y",
    errors="coerce"
)

In [33]:
historical_performance.head()

,abn_product_identifier,abn_investment_menu_identifier,abn_investment_option_identifier,fees_and_costs_arrangement_identifier,period_end_date,product_name,product_type,product_phase,investment_menu_name,investment_menu_type,...,investment_option_category,life_cycle_name,fund_name,rse_abn,trustee_name,fund_public_offer_status,open_to_new_mems,return_measurement_comparison_percent,return_investment_five_year_volatility_comparison_percent,return_investment_ten_year_volatility_comparison_percent
0,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,2014-12-31,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.026640,0.048896,NaN
1,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,2015-03-31,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.057225,0.046906,NaN
2,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,2015-06-30,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.016261,0.044379,NaN
3,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,2015-09-30,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,-0.017265,0.046924,NaN
4,66628776348-66628776348908,66628776348-CSMYSUPER,66628776348-CSPJ,1,2015-12-31,MySuper,MySuper Product,Accumulation,MySuper,Generic,...,Multi Sector,NaN,Christian Super,66628776348,NaN,Public Offer,Y,0.024534,0.047364,NaN
